# Script 8: Finalize Output Dataset by Joining All Signals

In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

In [2]:
cleaned = (
    session.table("with_on_topic_label")
    .select("url", "is_on_topic")
)

narrative_intent = (
    session.table("with_narrative_intent_label")
    .select("url", "narrative_intent_label")
)

technical_terms = (
    session.table("with_cleaned_technical_terms")
    .select("url", "cleaned_terms")
)

has_code = (
    session.table("with_features")
    .select("url", "has_code")
)

technical_complexity = (
    session.table("with_complexity_label")
    .select("url", "complexity_label")
)

with_topic_labels = (
    session.table("with_topic_labels")
    .select("url", "topic_name")
)

with_subtopic_labels = (
    session.table("with_subtopic_labels")
    .select("url", "sub_topic_name")
)

In [ ]:
finalized = (
    cleaned
    .join(narrative_intent, on="url", how="left")
    .join(technical_terms, on="url", how="left")
    .join(has_code, on="url", how="left")
    .join(technical_complexity, on="url", how="left")
    .join(with_topic_labels, on="url", how="left")
    .join(with_subtopic_labels, on="url", how="left")
)
finalized.write.save_as_table("with_final_labels", mode="overwrite")
finalized.write.parquet("with_final_labels.parquet", mode="overwrite")

In [4]:
session.stop()